# Graficos de metricas de execucao

Este notebook le os relatorios gerados por:

```bash
make profile-chunk-data
```

ou:

```bash
make profile-command COMMAND="poetry run python -B src/rag/index_hoi4_qdrant.py"
```

Ele procura execucoes em `metrics/*/samples.csv` e gera graficos em `avaliacao/outputs/metricas/`.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 100)

sns.set_theme(
    context="notebook",
    style="whitegrid",
    palette="viridis",
    font_scale=1.05,
)
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "#fbfbf7"

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Makefile").exists() and (candidate / "scripts" / "monitor_command.py").exists():
            return candidate
    raise FileNotFoundError("Nao encontrei a raiz do projeto a partir do diretorio atual.")

ROOT = find_project_root(Path.cwd())
METRICS_DIR = ROOT / "metrics"
OUTPUT_DIR = ROOT / "avaliacao" / "outputs" / "metricas"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROOT, METRICS_DIR, OUTPUT_DIR

## 1. Descobrir execucoes disponiveis

In [ ]:
def load_summary(run_dir: Path) -> dict:
    summary_path = run_dir / "summary.json"
    if not summary_path.exists():
        return {}
    with summary_path.open("r", encoding="utf-8") as file:
        return json.load(file)

def discover_runs(metrics_dir: Path) -> pd.DataFrame:
    rows = []
    if not metrics_dir.exists():
        return pd.DataFrame(rows)

    for samples_path in sorted(metrics_dir.glob("*/samples.csv")):
        run_dir = samples_path.parent
        summary = load_summary(run_dir)
        rows.append({
            "run_id": run_dir.name,
            "run_dir": run_dir,
            "samples_path": samples_path,
            "summary_path": run_dir / "summary.json",
            "stdout_path": run_dir / "stdout.log",
            "stderr_path": run_dir / "stderr.log",
            "started_at": summary.get("started_at"),
            "finished_at": summary.get("finished_at"),
            "elapsed_seconds": summary.get("elapsed_seconds"),
            "returncode": summary.get("returncode"),
            "peak_process_cpu_percent": summary.get("peak_process_cpu_percent"),
            "peak_system_cpu_percent": summary.get("peak_system_cpu_percent"),
            "peak_rss_mb": summary.get("peak_rss_mb"),
            "max_disk_read_mb": summary.get("max_disk_read_mb"),
            "max_disk_write_mb": summary.get("max_disk_write_mb"),
            "gpu_available": summary.get("gpu_available"),
            "peak_gpu_util_percent": summary.get("peak_gpu_util_percent"),
            "peak_gpu_memory_used_mb": summary.get("peak_gpu_memory_used_mb"),
            "command": " ".join(summary.get("command", [])),
        })

    runs = pd.DataFrame(rows)
    if not runs.empty:
        runs["started_at"] = pd.to_datetime(runs["started_at"], errors="coerce")
        runs["finished_at"] = pd.to_datetime(runs["finished_at"], errors="coerce")
        runs = runs.sort_values("started_at", na_position="last").reset_index(drop=True)
    return runs

runs = discover_runs(METRICS_DIR)
if runs.empty:
    print("Nenhuma execucao encontrada em metrics/. Gere dados com: make profile-chunk-data")
else:
    display(runs[["run_id", "started_at", "elapsed_seconds", "returncode", "peak_rss_mb", "peak_process_cpu_percent", "command"]])

## 2. Escolher execucao

Por padrao, o notebook usa a execucao mais recente. Para analisar outra, altere `RUN_INDEX`.

In [ ]:
RUN_INDEX = -1

if runs.empty:
    samples = pd.DataFrame()
    selected_run = None
else:
    selected_run = runs.iloc[RUN_INDEX]
    samples = pd.read_csv(selected_run["samples_path"])
    samples["timestamp"] = pd.to_datetime(samples["timestamp"], errors="coerce")
    samples = samples.sort_values("elapsed_seconds").reset_index(drop=True)
    print(f"Execucao selecionada: {selected_run['run_id']}")
    display(samples.head())

## 3. Funcoes de grafico

In [ ]:
def safe_filename(value: str) -> str:
    value = re.sub(r"[^a-zA-Z0-9._-]+", "-", value)
    return value.strip("-") or "grafico"

def save_current_figure(name: str) -> Path:
    path = OUTPUT_DIR / f"{safe_filename(name)}.png"
    plt.savefig(path, dpi=180, bbox_inches="tight")
    return path

def numeric_series(df: pd.DataFrame, column: str) -> pd.Series:
    return pd.to_numeric(df[column], errors="coerce")

def plot_line(df: pd.DataFrame, y: str, title: str, ylabel: str, filename: str, color=None) -> None:
    if df.empty or y not in df.columns:
        print(f"Sem dados para {y}.")
        return

    plot_df = df[["elapsed_seconds", y]].copy()
    plot_df[y] = numeric_series(plot_df, y)
    plot_df = plot_df.dropna(subset=[y])
    if plot_df.empty:
        print(f"Coluna {y} sem valores numericos.")
        return

    fig, ax = plt.subplots(figsize=(13, 4.5))
    sns.lineplot(
        data=plot_df,
        x="elapsed_seconds",
        y=y,
        marker="o",
        linewidth=2.2,
        markersize=5,
        color=color,
        ax=ax,
    )
    ax.fill_between(plot_df["elapsed_seconds"], plot_df[y], alpha=0.12, color=color)
    ax.set_title(title, fontsize=14, weight="bold", loc="left")
    ax.set_xlabel("Tempo desde o inicio da execucao (s)")
    ax.set_ylabel(ylabel)
    sns.despine(ax=ax)
    path = save_current_figure(filename)
    plt.show()
    print(f"Salvo em: {path}")


## 4. Linha do tempo da execucao selecionada

In [ ]:
if selected_run is not None:
    run_id = selected_run["run_id"]
    palette = sns.color_palette("viridis", 7)
    plot_line(samples, "process_cpu_percent", f"CPU do processo - {run_id}", "CPU do processo (%)", f"{run_id}-cpu-processo", color=palette[0])
    plot_line(samples, "system_cpu_percent", f"CPU do sistema - {run_id}", "CPU do sistema (%)", f"{run_id}-cpu-sistema", color=palette[1])
    plot_line(samples, "rss_mb", f"Memoria RSS - {run_id}", "Memoria RSS (MB)", f"{run_id}-memoria-rss", color=palette[2])
    plot_line(samples, "disk_read_mb", f"Leitura de disco - {run_id}", "Leitura acumulada (MB)", f"{run_id}-disco-leitura", color=palette[3])
    plot_line(samples, "disk_write_mb", f"Escrita de disco - {run_id}", "Escrita acumulada (MB)", f"{run_id}-disco-escrita", color=palette[4])


In [ ]:
if selected_run is not None and "gpu_available" in samples.columns and samples["gpu_available"].astype(str).str.lower().eq("true").any():
    palette = sns.color_palette("rocket", 4)
    plot_line(samples, "gpu_util_percent_max", f"GPU util - {run_id}", "GPU util max (%)", f"{run_id}-gpu-util", color=palette[1])
    plot_line(samples, "gpu_memory_used_mb_sum", f"GPU memoria - {run_id}", "GPU memoria usada total (MB)", f"{run_id}-gpu-memoria", color=palette[2])
else:
    print("Sem GPU detectada nas amostras desta execucao.")


## 5. Painel combinado

Este painel facilita enxergar correlacoes entre CPU, memoria, disco e GPU no mesmo intervalo de tempo.

In [ ]:
if selected_run is not None and not samples.empty:
    metrics = [
        ("process_cpu_percent", "CPU processo (%)"),
        ("system_cpu_percent", "CPU sistema (%)"),
        ("rss_mb", "RSS (MB)"),
        ("disk_read_mb", "Disco leitura (MB)"),
        ("disk_write_mb", "Disco escrita (MB)"),
        ("gpu_util_percent_max", "GPU util (%)"),
        ("gpu_memory_used_mb_sum", "GPU memoria (MB)"),
    ]
    available = [(column, label) for column, label in metrics if column in samples.columns and pd.to_numeric(samples[column], errors="coerce").notna().any()]

    panel_rows = []
    for column, label in available:
        current = samples[["elapsed_seconds", column]].copy()
        current[column] = pd.to_numeric(current[column], errors="coerce")
        current = current.dropna(subset=[column])
        current = current.rename(columns={column: "value"})
        current["metric"] = label
        panel_rows.append(current)

    panel_df = pd.concat(panel_rows, ignore_index=True) if panel_rows else pd.DataFrame()
    if not panel_df.empty:
        grid = sns.relplot(
            data=panel_df,
            x="elapsed_seconds",
            y="value",
            row="metric",
            kind="line",
            marker="o",
            height=2.1,
            aspect=5.8,
            facet_kws={"sharey": False, "sharex": True},
            linewidth=1.8,
        )
        grid.set_axis_labels("Tempo desde o inicio da execucao (s)", "")
        grid.set_titles(row_template="{row_name}")
        grid.figure.suptitle(f"Painel de metricas - {selected_run['run_id']}", fontsize=15, weight="bold", x=0.02, ha="left")
        grid.figure.subplots_adjust(top=0.94, hspace=0.32)
        path = OUTPUT_DIR / f"{safe_filename(selected_run['run_id'])}-painel.png"
        grid.figure.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        print(f"Salvo em: {path}")


## 6. Eventos de stdout/stderr

Use esta tabela para cruzar eventos do comando com os graficos de metricas.

In [ ]:
def read_log_events(path: Path, stream: str) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    rows = []
    pattern = re.compile(r"^(\S+)\s+\[(.*?)\]\s?(.*)$")
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.rstrip("\n")
            match = pattern.match(line)
            if match:
                rows.append({
                    "stream": stream,
                    "line_number": line_number,
                    "timestamp": match.group(1),
                    "label": match.group(2),
                    "message": match.group(3),
                })
            else:
                rows.append({"stream": stream, "line_number": line_number, "timestamp": None, "label": None, "message": line})
    events = pd.DataFrame(rows)
    if not events.empty:
        events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")
    return events

if selected_run is not None:
    stdout_events = read_log_events(selected_run["stdout_path"], "stdout")
    stderr_events = read_log_events(selected_run["stderr_path"], "stderr")
    events = pd.concat([stdout_events, stderr_events], ignore_index=True)
    events = events.sort_values(["timestamp", "stream", "line_number"], na_position="last").reset_index(drop=True)
    events_path = OUTPUT_DIR / f"{safe_filename(selected_run['run_id'])}-eventos.csv"
    events.to_csv(events_path, index=False)
    display(events.head(100))
    print(f"Eventos salvos em: {events_path}")

## 7. Comparar execucoes

Use esta secao para ver se uma mudanca reduziu tempo, memoria, CPU ou IO em comparacao com rodadas anteriores.

In [ ]:
if runs.empty:
    print("Sem execucoes para comparar.")
else:
    comparison_columns = [
        "run_id",
        "started_at",
        "elapsed_seconds",
        "peak_rss_mb",
        "peak_process_cpu_percent",
        "peak_system_cpu_percent",
        "max_disk_read_mb",
        "max_disk_write_mb",
        "peak_gpu_util_percent",
        "peak_gpu_memory_used_mb",
        "returncode",
    ]
    comparison = runs[comparison_columns].copy()
    display(comparison)

In [ ]:
if not runs.empty:
    chart_columns = [
        ("elapsed_seconds", "Tempo total (s)"),
        ("peak_rss_mb", "Pico memoria RSS (MB)"),
        ("max_disk_read_mb", "Max leitura disco (MB)"),
        ("max_disk_write_mb", "Max escrita disco (MB)"),
    ]
    comparison_rows = []
    for column, label in chart_columns:
        current = runs[["run_id", column]].copy()
        current[column] = pd.to_numeric(current[column], errors="coerce")
        current = current.rename(columns={column: "value"})
        current["metric"] = label
        comparison_rows.append(current)

    comparison_plot_df = pd.concat(comparison_rows, ignore_index=True)
    grid = sns.catplot(
        data=comparison_plot_df,
        x="run_id",
        y="value",
        row="metric",
        kind="bar",
        height=2.5,
        aspect=5.5,
        sharey=False,
        palette="viridis",
        hue="run_id",
        legend=False,
    )
    grid.set_axis_labels("Execucao", "")
    grid.set_titles(row_template="{row_name}")
    for ax in grid.axes.flat:
        ax.tick_params(axis="x", rotation=70)
        sns.despine(ax=ax)
    grid.figure.suptitle("Comparacao entre execucoes", fontsize=15, weight="bold", x=0.02, ha="left")
    grid.figure.subplots_adjust(top=0.92, hspace=0.55)
    path = OUTPUT_DIR / "comparacao-execucoes.png"
    grid.figure.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Salvo em: {path}")


## 8. Exportar serie consolidada

Este CSV combina as amostras de todas as execucoes e facilita analises externas.

In [ ]:
all_samples = []
for _, run in runs.iterrows():
    current = pd.read_csv(run["samples_path"])
    current.insert(0, "run_id", run["run_id"])
    current.insert(1, "run_started_at", run["started_at"])
    all_samples.append(current)

if all_samples:
    all_samples_df = pd.concat(all_samples, ignore_index=True)
    consolidated_path = OUTPUT_DIR / "samples_consolidados.csv"
    all_samples_df.to_csv(consolidated_path, index=False)
    display(all_samples_df.head())
    print(f"Serie consolidada salva em: {consolidated_path}")
else:
    print("Sem amostras para exportar.")